In [39]:

import requests
import pandas as pd

url = "https://api.census.gov/data/2024/acs/acs5"
params = {
    "get": "NAME,B19013_001E,B27001_001E,B01003_001E", #name,median household income, health insurance coverage, total population
    "for": "county:*",          # all counties
    "in": "state:*",            # all states
    "key": '750defc8aa4a25457ddbe3a4f3b4bf27e31595b9'
}
r = requests.get(url, params=params)
df = pd.DataFrame(r.json()[1:], columns=r.json()[0])

In [40]:
df.to_csv("census_county_data_raw.csv", index=False)

In [41]:
# Create a combined FIPS code column for merging with other datasets later
df["geoid"] = df["state"] + df["county"]
df["geoid"] = df["state"].str.zfill(2) + df["county"].str.zfill(3)

# Convert numeric columns from strings (Census API returns everything as strings)
df["B19013_001E"] = pd.to_numeric(df["B19013_001E"], errors="coerce")
df["B27001_001E"] = pd.to_numeric(df["B27001_001E"], errors="coerce")
df["B01003_001E"] = pd.to_numeric(df["B01003_001E"], errors="coerce")

# Rename to something readable
df = df.rename(columns={
    "B19013_001E": "median_household_income",
    "B27001_001E": "health_insurance_total",
    "B01003_001E": "total_population"
})


In [42]:
df.to_csv("census_county_data.csv", index=False)

In [43]:
import requests
import zipfile
import io
import pandas as pd

# Download and unzip in memory — no need to save the zip file
url = "https://www2.census.gov/programs-surveys/sahie/datasets/time-series/estimates-acs/sahie-2022-csv.zip"

response = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(response.content))

# Check what's inside the zip
print(z.namelist())


['sahie_2022.csv']


In [44]:
# Read the raw lines to find where the actual data starts
with z.open('sahie_2022.csv') as f:
    lines = f.readlines()

# Print the first 85 lines to find where the header row is
for i, line in enumerate(lines[:85]):
    print(i, line.decode('utf-8').strip())

0 Filename:  sahie_2022.csv
1 Created:   12JUL24  02:44
2 Description:
3 
4 Model-based Small Area Health Insurance Estimates (SAHIE) for Counties and States, 2022
5 Source: U.S. Census Bureau, Small Area Health Insurance Estimates (SAHIE) Program, July 2024, Project No. P-6000007 / Approval CBDRB FY24-0310
6 
7 NOTE: VERY LONG .CSV FILES NOT ALWAYS IMPORTED INTO EXCEL CORRECTLY - CHECK FOR TRUNCATION
8 
9 File Layout and Definitions:
10 
11 Variable      Format      Description
12 year            4      Year of Estimate
13 version         8      Release Version
14 Blank   : YEAR other than 2013, Only Version
15 Original: 2013 only, Original Version
16 Updated : 2013 only, Updated Version (May 2016)
17 statefips       2      Unique FIPS code for each state
18 countyfips      3      Unique FIPS code for each county within a state
19 geocat          2      Geography category
20 40 - State geographic identifier
21 50 - County geographic identifier
22 agecat          1      Age category
23

In [45]:
df_sahie = pd.read_csv(z.open('sahie_2022.csv'), skiprows=84)
for col in ["PCTUI", "NUI", "NIPR", "statefips", "countyfips", "agecat", "iprcat", "sexcat", "racecat"]:
    df_sahie[col] = pd.to_numeric(df_sahie[col], errors="coerce")

C:\Users\17063\AppData\Local\Temp\ipykernel_46484\4102004643.py:1: DtypeWarning: Columns (0: NIPR, 1: nipr_moe, 2: NUI, 3: nui_moe, 4: NIC, 5: nic_moe, 6: PCTUI, 7: pctui_moe, 8: PCTIC, 9: pctic_moe, 10: PCTELIG, 11: pctelig_moe, 12: PCTLIIC, 13: pctliic_moe) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sahie = pd.read_csv(z.open('sahie_2022.csv'), skiprows=84)


In [46]:
df_sahie.to_csv("sahie_county_data_raw.csv", index=False)

In [47]:
print(df_sahie.columns.tolist())

['year', 'version', 'statefips', 'countyfips', 'geocat', 'agecat', 'racecat', 'sexcat', 'iprcat', 'NIPR', 'nipr_moe', 'NUI', 'nui_moe', 'NIC', 'nic_moe', 'PCTUI', 'pctui_moe', 'PCTIC', 'pctic_moe', 'PCTELIG', 'pctelig_moe', 'PCTLIIC', 'pctliic_moe', 'state_name', 'county_name', 'Unnamed: 25']


In [48]:
# Filter to all ages, all incomes, both sexes, all races
df_sahie = df_sahie[
    (df_sahie["agecat"] == 0) &
    (df_sahie["iprcat"] == 0) &
    (df_sahie["sexcat"] == 0) &
    (df_sahie["racecat"] == 0)
]

# Build geoid and uninsured rate
df_sahie["geoid"] = df_sahie["statefips"].astype(str).str.zfill(2) + \
                    df_sahie["countyfips"].astype(str).str.zfill(3)
df_sahie["uninsured_rate"] = df_sahie["PCTUI"] / 100

# Drop statewide rows (countyfips == 0)
df_sahie = df_sahie[df_sahie["countyfips"] != 0]

In [49]:
df_sahie = df_sahie[["geoid", "uninsured_rate", "NUI", "NIPR"]]

In [50]:
df_sahie = df_sahie.rename(columns={
    "NUI": "uninsured_count",
    "NIPR": "total_population"
})

In [51]:
df_sahie.to_csv("sahie_county_data.csv", index=False)

In [52]:
# HPSA file
url = "https://data.hrsa.gov/DataDownload/DD_Files/BCD_HPSA_FCT_DET_PC.csv"
response = requests.get(url)

df_hpsa = pd.read_csv(io.BytesIO(response.content))

C:\Users\17063\AppData\Local\Temp\ipykernel_46484\3084611216.py:5: DtypeWarning: Columns (0: BHCMIS Organization Identification Number) have mixed types. Specify dtype option on import or set low_memory=False.
  df_hpsa = pd.read_csv(io.BytesIO(response.content))


In [53]:
print(df_hpsa.shape)
print(df_hpsa.columns.tolist())

(74920, 66)
['HPSA Name', 'HPSA ID', 'Designation Type', 'HPSA Discipline Class', 'HPSA Score', 'PC MCTA Score', 'Primary State Abbreviation', 'HPSA Status', 'HPSA Designation Date', 'HPSA Designation Last Update Date', 'Metropolitan Indicator', 'HPSA Geography Identification Number', 'HPSA Degree of Shortage', 'Withdrawn Date', 'HPSA FTE', 'HPSA Designation Population', '% of Population Below 100% Poverty', 'HPSA Formal Ratio', 'HPSA Population Type', 'Rural Status', 'Longitude', 'Latitude', 'BHCMIS Organization Identification Number', 'Break in Designation', 'Common County Name', 'Common Postal Code', 'Common Region Name', 'Common State Abbreviation', 'Common State County FIPS Code', 'Common State FIPS Code', 'Common State Name', 'County Equivalent Name', 'County or County Equivalent Federal Information Processing Standard Code', 'Discipline Class Number', 'HPSA Address', 'HPSA City', 'HPSA Component Name', 'HPSA Component Source Identification Number', 'HPSA Component State Abbrevia

In [55]:
df_hpsa.to_csv("hpsa_county_data_raw.csv", index=False)

In [ ]:
# Filter to active primary care designations
df_hpsa = df_hpsa[df_hpsa["HPSA Status"] == "Designated"]
df_hpsa = df_hpsa[df_hpsa["HPSA Discipline Class"] == "Primary Care"]

# Keep useful columns
df_hpsa = df_hpsa[[
    "Common State FIPS Code",
    "State and County Federal Information Processing Standard Code",
    "Common County Name",
    "Common State Name",
    "HPSA Name",
    "HPSA Status",
    "HPSA Score",
    "Designation Type",
    "HPSA Designation Population",
    "HPSA Estimated Underserved Population",
    "HPSA Formal Ratio",        # provider-to-population ratio
    "% of Population Below 100% Poverty",
    "Rural Status",
    "Metropolitan Indicator"
]]

# The full 5-digit FIPS is already combined in one column
df_hpsa = df_hpsa.rename(columns={
    "State and County Federal Information Processing Standard Code": "geoid",
    "HPSA Score": "hpsa_score",
    "HPSA Formal Ratio": "provider_ratio",
    "% of Population Below 100% Poverty": "pct_below_poverty",
    "HPSA Designation Population": "hpsa_population",
    "HPSA Estimated Underserved Population": "underserved_population",
    "Rural Status": "rural_status",
    "Metropolitan Indicator": "metro_indicator",
    "Common County Name": "county_name",
    "Common State Name": "state_name"
})

# Zero-pad geoid just in case
df_hpsa["geoid"] = df_hpsa["geoid"].astype(str).str.zfill(5)

# Aggregate to one row per county
df_hpsa_county = df_hpsa.groupby("geoid").agg(
    hpsa_score_max=("hpsa_score", "max"), #composite HPSA score.
    hpsa_designation_count=("HPSA Name", "count"), #how many shortage areas in a county
    pct_below_poverty=("pct_below_poverty", "max")
).reset_index()
df_hpsa_county["hpsa_designated"] = (df_hpsa_county["hpsa_score_max"] > 0).astype(int)

In [61]:
df_hpsa.to_csv("hpsa_county_data.csv", index=False)
df_hpsa_county.to_csv("hpsa_county_composite_score.csv", index=False)

In [63]:
url = "https://data.hrsa.gov/DataDownload/DD_Files/UDS_2023_Data.zip"
response = requests.get(url)

# Check what you actually got back
print(response.headers.get("Content-Type"))
print(response.status_code)

text/html
404


In [7]:
import requests
import pandas as pd

url = "https://data.cdc.gov/resource/swc5-untb.json"

all_records = []
limit = 50000
offset = 0

while True:
    params = {
        "$limit": limit,
        "$offset": offset
    }
    
    response = requests.get(url, params=params)
    batch = response.json()
    
    if not batch:
        break
        
    all_records.extend(batch)
    print(f"Fetched {len(all_records)} records so far...")
    
    if len(batch) < limit:
        break
        
    offset += limit

df_places = pd.DataFrame(all_records)
print(f"\nTotal records: {df_places.shape}")
print(df_places.columns.tolist())

Fetched 50000 records so far...
Fetched 100000 records so far...
Fetched 150000 records so far...
Fetched 200000 records so far...
Fetched 229298 records so far...

Total records: (229298, 24)
['year', 'stateabbr', 'statedesc', 'locationname', 'datasource', 'category', 'measure', 'data_value_unit', 'data_value_type', 'data_value', 'low_confidence_limit', 'high_confidence_limit', 'totalpopulation', 'totalpop18plus', 'locationid', 'categoryid', 'measureid', 'datavaluetypeid', 'short_question_text', 'geolocation', ':@computed_region_hjsp_umg2', ':@computed_region_skr5_azej', 'data_value_footnote_symbol', 'data_value_footnote']


In [8]:
df_places.to_csv("cdc_prevalence_raw.csv", index=False)

In [9]:
# See what health measures are available
print(df_places["measureid"].unique())

<StringArray>
[   'ARTHRITIS',      'CASTHMA',       'STROKE',   'DISABILITY',
      'OBESITY',     'DIABETES',   'DEPRESSION',      'HEARING',
        'BINGE',       'BPHIGH',       'VISION',     'MOBILITY',
     'SELFCARE',        'MHLTH',     'HIGHCHOL',        'PHLTH',
 'COLON_SCREEN',          'LPA',         'COPD',    'INDEPLIVE',
          'CHD',     'MAMMOUSE',      'CHECKUP',        'BPMED',
     'CSMOKING',   'HOUSINSECU',  'SHUTUTILITY',   'FOODINSECU',
    'TEETHLOST',        'GHLTH',     'LACKTRPT',       'CANCER',
   'EMOTIONSPT',       'DENTAL',   'LONELINESS',    'FOODSTAMP',
      'ACCESS2',    'COGNITION',   'CHOLSCREEN',        'SLEEP']
Length: 40, dtype: str


In [10]:
measures = [
    "ACCESS2",   # lack of health insurance (redundant with SAHIE but good validation)
    "DIABETES",  # diabetes prevalence
    "HIGHCHOL",  # high cholesterol
    "BPHIGH",    # high blood pressure
    "CASTHMA",   # asthma
    "CHECKUP",   # no routine checkup in past year — very relevant for care desert validation
    "DENTAL",    # no dental visit
]

df_places = df_places[df_places["measureid"].isin(measures)]

In [11]:
print(df_places.shape)        # how many rows and columns
print(df_places.columns.tolist())  # column names
print(df_places.head())       # first few rows
print(df_places["measureid"].value_counts())  # what health measures are in there

(41786, 24)
['year', 'stateabbr', 'statedesc', 'locationname', 'datasource', 'category', 'measure', 'data_value_unit', 'data_value_type', 'data_value', 'low_confidence_limit', 'high_confidence_limit', 'totalpopulation', 'totalpop18plus', 'locationid', 'categoryid', 'measureid', 'datavaluetypeid', 'short_question_text', 'geolocation', ':@computed_region_hjsp_umg2', ':@computed_region_skr5_azej', 'data_value_footnote_symbol', 'data_value_footnote']
    year stateabbr statedesc locationname datasource         category  \
1   2023        AR  Arkansas       Fulton      BRFSS  Health Outcomes   
9   2023        CO  Colorado         Lake      BRFSS  Health Outcomes   
13  2023        FL   Florida      Brevard      BRFSS  Health Outcomes   
14  2023        FL   Florida       Citrus      BRFSS  Health Outcomes   
57  2023        KS    Kansas      Haskell      BRFSS  Health Outcomes   

                            measure data_value_unit   data_value_type  \
1       Current asthma among adults  

In [12]:
# Keep just what you need
df_places = df_places[["locationid", "measureid", "data_value"]]

df_places["data_value"] = pd.to_numeric(df_places["data_value"], errors="coerce")

# Pivot to one row per county
df_places_wide = df_places.pivot_table(
    index="locationid",
    columns="measureid",
    values="data_value"
).reset_index()

# Rename the index column to geoid for merging
df_places_wide.columns.name = None
df_places_wide = df_places_wide.rename(columns={"locationid": "geoid"})
df_places_wide["geoid"] = df_places_wide["geoid"].astype(str).str.zfill(5)

In [13]:
df_places_wide.shape

(3144, 8)

In [14]:
df_places_wide = df_places_wide.rename(columns={
    "ACCESS2": "uninsured_pct",
    "BPHIGH": "high_bp_pct",
    "CASTHMA": "asthma_pct",
    "CHECKUP": "no_check_up_pct",
    "DENTAL": "no_dental_visit_pct",
    "DIABETES": "diabetes_pct",
    "HIGHCHOL": "high_cholesterol_pct"
})

In [15]:
df_places_wide.to_csv("cdc_prevalence.csv", index=False)

In [68]:
# Tract-level endpoint
url = "https://data.cdc.gov/resource/cwsq-ngmh.json"

all_data = []
limit = 50000
offset = 0

while True:
    params = {"$limit": limit, "$offset": offset}
    response = requests.get(url, params=params)
    batch = response.json()
    
    if not batch:
        break
    
    all_data.extend(batch)
    offset += limit
    print(f"Downloaded {offset} rows...")

df_places_tract = pd.DataFrame(all_data)

Downloaded 50000 rows...
Downloaded 100000 rows...
Downloaded 150000 rows...
Downloaded 200000 rows...
Downloaded 250000 rows...
Downloaded 300000 rows...
Downloaded 350000 rows...
Downloaded 400000 rows...
Downloaded 450000 rows...
Downloaded 500000 rows...
Downloaded 550000 rows...
Downloaded 600000 rows...
Downloaded 650000 rows...
Downloaded 700000 rows...
Downloaded 750000 rows...
Downloaded 800000 rows...
Downloaded 850000 rows...
Downloaded 900000 rows...
Downloaded 950000 rows...
Downloaded 1000000 rows...
Downloaded 1050000 rows...
Downloaded 1100000 rows...
Downloaded 1150000 rows...
Downloaded 1200000 rows...
Downloaded 1250000 rows...
Downloaded 1300000 rows...
Downloaded 1350000 rows...
Downloaded 1400000 rows...
Downloaded 1450000 rows...
Downloaded 1500000 rows...
Downloaded 1550000 rows...
Downloaded 1600000 rows...
Downloaded 1650000 rows...
Downloaded 1700000 rows...
Downloaded 1750000 rows...
Downloaded 1800000 rows...
Downloaded 1850000 rows...
Downloaded 1900000 ro

In [78]:
df_places_tract.to_csv("cdc_prevalence_tractlvl_raw.csv", index=False)

In [75]:
df_places_tract.head(5)

,year,stateabbr,statedesc,countyname,countyfips,locationname,datasource,category,measure,data_value_unit,...,high_confidence_limit,totalpopulation,totalpop18plus,geolocation,locationid,categoryid,measureid,datavaluetypeid,short_question_text,:@computed_region_skr5_azej
0,2023,AL,Alabama,Jefferson,01073,01073010900,BRFSS,Disability,Any disability among adults,%,...,49.0,4719,3410,"{'type': 'Point', 'coordinates': [-86.7705771,...",01073010900,DISABLT,DISABILITY,CrdPrv,Any Disability,1583
1,2023,AL,Alabama,Jefferson,01073,01073011207,BRFSS,Health Outcomes,Depression among adults,%,...,25.9,5103,3684,"{'type': 'Point', 'coordinates': [-86.6742325,...",01073011207,HLTHOUT,DEPRESSION,CrdPrv,Depression,1583
2,2023,AL,Alabama,Lauderdale,01077,01077010100,BRFSS,Disability,Any disability among adults,%,...,46.2,2278,2093,"{'type': 'Point', 'coordinates': [-87.6598801,...",01077010100,DISABLT,DISABILITY,CrdPrv,Any Disability,1584
3,2023,AL,Alabama,Lee,01081,01081040202,BRFSS,Health Outcomes,Arthritis among adults,%,...,26.2,3084,2459,"{'type': 'Point', 'coordinates': [-85.4521355,...",01081040202,HLTHOUT,ARTHRITIS,CrdPrv,Arthritis,1586
4,2023,AL,Alabama,Lee,01081,01081040300,BRFSS,Health Outcomes,Obesity among adults,%,...,42.2,2576,2188,"{'type': 'Point', 'coordinates': [-85.4668576,...",01081040300,HLTHOUT,OBESITY,CrdPrv,Obesity,1586


In [79]:
measures = [
    "ACCESS2",   # lack of health insurance (redundant with SAHIE but good validation)
    "DIABETES",  # diabetes prevalence
    "HIGHCHOL",  # high cholesterol
    "BPHIGH",    # high blood pressure
    "CASTHMA",   # asthma
    "CHECKUP",   # no routine checkup in past year — very relevant for care desert validation
    "DENTAL",    # no dental visit
]

df_places_tract = df_places_tract[df_places_tract["measureid"].isin(measures)]

In [80]:
# Keep what you need
df_places_tract = df_places_tract[["locationid", "measureid", "data_value"]]

df_places_tract["data_value"] = pd.to_numeric(df_places_tract["data_value"], errors="coerce")

# Pivot wide
df_places_tract_wide = df_places_tract.pivot_table(
    index="locationid",
    columns="measureid",
    values="data_value"
).reset_index()

df_places_tract_wide.columns.name = None
df_places_tract_wide = df_places_tract_wide.rename(columns={"locationid": "geoid"})
df_places_tract_wide["geoid"] = df_places_tract_wide["geoid"].astype(str).str.zfill(11)

# Add county geoid for linking back up
df_places_tract_wide["geoid_county"] = df_places_tract_wide["geoid"].str[:5]

In [82]:
df_places_tract_wide = df_places_tract_wide.rename(columns={
    "ACCESS2": "uninsured_pct",
    "BPHIGH": "high_bp_pct",
    "CASTHMA": "asthma_pct",
    "CHECKUP": "no_check_up_pct",
    "DENTAL": "no_dental_visit_pct",
    "DIABETES": "diabetes_pct",
    "HIGHCHOL": "high_cholesterol_pct"
})

In [83]:
df_places_tract_wide.head(5)

,geoid,uninsured_pct,high_bp_pct,asthma_pct,no_check_up_pct,no_dental_visit_pct,diabetes_pct,high_cholesterol_pct,geoid_county
0,01001020100,9.7,41.4,10.2,80.1,57.3,13.3,41.9,01001
1,01001020200,10.6,45.9,10.8,81.8,56.1,15.8,39.7,01001
2,01001020300,10.6,42.9,10.7,80.6,57.7,13.9,41.8,01001
3,01001020400,7.9,42.4,9.6,81.7,61.1,12.8,43.4,01001
4,01001020501,6.2,37.2,9.3,81.3,66.3,10.2,40.5,01001


In [84]:
df_places_tract_wide.to_csv("cdc_prevalence_tractlvl.csv", index=False)

In [4]:
import requests
import pandas as pd
import io

# HRSA updated the URL structure — now uses year-versioned filenames
# Most recent as of March 2025 update:
urls_to_try = [
    "https://data.hrsa.gov/DataDownload/DD_Files/AHRF_2023-2024_COUNTY.xlsx",
    "https://data.hrsa.gov/DataDownload/DD_Files/AHRF_2022-2023_COUNTY.xlsx",
    "https://data.hrsa.gov//DataDownload/AHRF/AHRF_2023-2024.xlsx",
]

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})  # Some gov sites block default requests UA

df_ahrf = None
for url in urls_to_try:
    try:
        response = session.get(url, timeout=120)
        response.raise_for_status()
        # Sanity check — make sure we got Excel, not an HTML error page
        if "html" in response.headers.get("Content-Type", ""):
            print(f"Got HTML instead of Excel from: {url}")
            continue
        df_ahrf = pd.read_excel(io.BytesIO(response.content), engine="openpyxl")
        print(f"Success: {url}")
        print(df_ahrf.shape)
        break
    except Exception as e:
        print(f"Failed: {url} → {e}")

if df_ahrf is None:
    print("\nAll URLs failed. Manually check: https://data.hrsa.gov/data/download?data=AHRF#AHRF")

Failed: https://data.hrsa.gov/DataDownload/DD_Files/AHRF_2023-2024_COUNTY.xlsx → 404 Client Error: Not Found for url: https://data.hrsa.gov/DataDownload/DD_Files/AHRF_2023-2024_COUNTY.xlsx
Failed: https://data.hrsa.gov/DataDownload/DD_Files/AHRF_2022-2023_COUNTY.xlsx → 404 Client Error: Not Found for url: https://data.hrsa.gov/DataDownload/DD_Files/AHRF_2022-2023_COUNTY.xlsx
Failed: https://data.hrsa.gov//DataDownload/AHRF/AHRF_2023-2024.xlsx → 404 Client Error: Not Found for url: https://data.hrsa.gov//DataDownload/AHRF/AHRF_2023-2024.xlsx

All URLs failed. Manually check: https://data.hrsa.gov/data/download?data=AHRF#AHRF


In [5]:
df_ahrf.head(5)

AttributeError: 'NoneType' object has no attribute 'head'